In [1]:
%pip install seaborn lightgbm mlflow

  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sagemaker 2.235.2 requires google-pasta, which is not installed.
sagemaker 2.235.2 requires pathos, which is not installed.
sagemaker 2.235.2 requires tblib<4,>=1.7.0, which is not installed.
sagemaker-training 4.9.0 requires protobuf<=3.20.3,>=3.9.2, but you have protobuf 6.33.6 which is incompatible.
sagemaker 2.235.2 requires attrs<24,>=23.1.0, but you have attrs 26.1.0 which is incompatible.
sagemaker 2.235.2 requires cloudpickle==2.2.1, but you have cloudpickle 3.1.2 which is incompatible.
sa

# modeling_ddb — DynamoDB + MLflow 통합 파이프라인

DynamoDB에서 config/data를 로드하고, 학습 결과를 DynamoDB와 MLflow에 저장합니다.
로컬 파일을 생성하지 않습니다.

## 데이터 흐름
```
DynamoDB (CONF / DATA#split)
        ↓ 로드
  LightGBM 학습
        ↓ 저장
DynamoDB (METRICS / MODEL / CHARTS / ...)
MLflow   (params / metrics / model / charts)
```

## 1. Import

In [13]:
import io, os, sys, json, yaml, math, base64, hashlib, pickle
import tempfile, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from pathlib import Path
from decimal import Decimal
from datetime import datetime
from uuid import uuid4
from boto3.dynamodb.conditions import Key

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, roc_curve,
)
import lightgbm as lgb
import mlflow, mlflow.lightgbm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('라이브러리 임포트 완료')

라이브러리 임포트 완료


## 2. DynamoDB 연결 및 타입 변환 헬퍼

### 테이블 키 구조
```
experiment_id (HASH)  |  entity_type (RANGE)
──────────────────────────────────────────────────────────
EXP#{user_id}#{project}#{experiment}  |  CONF / DATA#split
RUN#{user_id}#{project}#{experiment}#{run_id}  |  METRICS / MODEL#chunk_n / ...
```

In [14]:
TABLE_NAME = 'gsretail-mlops-edu-hjsong'
REGION     = 'us-east-1'
USER_ID    = 'hjsong'
PROJECT    = 'titanic-survival-prediction'
EXPERIMENT = 'tuned-hjsong-v2'

# DynamoDB 연결
ddb   = boto3.resource('dynamodb', region_name=REGION)
table = ddb.Table(TABLE_NAME)

# 키 생성
EXP_PK = f'EXP#{USER_ID}#{PROJECT}#{EXPERIMENT}'

print(f'Table        : {TABLE_NAME}')
print(f'experiment_id: {EXP_PK}')

Table        : gsretail-mlops-edu-hjsong
experiment_id: EXP#hjsong#titanic-survival-prediction#tuned-hjsong-v2


In [15]:
def to_ddb(obj):
    """float → Decimal (DynamoDB 저장 전)"""
    if isinstance(obj, float):   return Decimal(str(obj))
    elif isinstance(obj, bool):  return obj
    elif isinstance(obj, dict):  return {k: to_ddb(v) for k, v in obj.items()}
    elif isinstance(obj, list):  return [to_ddb(v) for v in obj]
    return obj

def from_ddb(obj):
    """Decimal → int/float (DynamoDB 로드 후)"""
    if isinstance(obj, Decimal):
        f = float(obj)
        return int(f) if f == int(f) else f
    elif isinstance(obj, dict):  return {k: from_ddb(v) for k, v in obj.items()}
    elif isinstance(obj, list):  return [from_ddb(v) for v in obj]
    return obj

print('to_ddb / from_ddb 헬퍼 정의 완료')

to_ddb / from_ddb 헬퍼 정의 완료


## 3. config 로드 (DynamoDB CONF 아이템)

In [16]:
# CONF 아이템 조회: entity_type = 'CONF'
resp      = table.get_item(Key={'experiment_id': EXP_PK, 'entity_type': 'CONF'})
conf_item = from_ddb(resp['Item'])

env_config   = conf_item['env_yml']
meta_config  = conf_item['meta_yml']
model_config = conf_item['model_yml']

VERSION    = meta_config['version']
ENV        = env_config['env']

print(f'Project   : {meta_config["project"]}')
print(f'Experiment: {meta_config["experiment"]}')
print(f'Algorithm : {model_config["algorithm"]["name"]}')

Project   : titanic-survival-prediction
Experiment: tuned-hjsong-v2
Algorithm : lightgbm


## 4. MLflow 설정

In [17]:
mlflow_config       = meta_config.get('mlflow', {})
MLFLOW_TRACKING_URI = mlflow_config.get('tracking_uri', '')
MLFLOW_EXPERIMENT   = mlflow_config.get('experiment_name', '')
MLFLOW_USERNAME     = mlflow_config.get('username', '')
MLFLOW_TAGS         = mlflow_config.get('tags', [])
SECRET_NAME         = mlflow_config.get('secret_name', '')

os.environ['MLFLOW_TRACKING_USERNAME'] = 'edu05'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'edu05Pass!234'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f'Tracking URI: {MLFLOW_TRACKING_URI}')
print(f'Experiment  : {MLFLOW_EXPERIMENT}')

Tracking URI: http://edu-mlflow-alb-697692344.us-east-1.elb.amazonaws.com
Experiment  : passenger-survival-classifier/tuned-hjsong-v2


In [18]:
# Parameters 태그 (papermill 주입용)
run_id     = None
output_dir = None

In [19]:
# Run ID 생성
RUN_DATE   = datetime.now().strftime('%Y%m%d')
ALGORITHM  = model_config['algorithm']['name']
SUFFIX     = model_config['algorithm']['suffix']
RUN_ID     = run_id if run_id else f'{RUN_DATE}_{ALGORITHM}_{SUFFIX}_{uuid4().hex[:8]}'
CREATED_AT = datetime.now().isoformat()

# RUN PK: RUN#{user_id}#{project}#{experiment}#{run_id}
RUN_PK = f'RUN#{USER_ID}#{PROJECT}#{EXPERIMENT}#{RUN_ID}'

# MLflow run_name 규칙: {project}__{experiment}__{user_id}__{algorithm}
MLFLOW_RUN_NAME = f'{PROJECT}__{EXPERIMENT}__{USER_ID}__{ALGORITHM}'

print(f'RUN_ID : {RUN_ID}')
print(f'RUN_PK : {RUN_PK}')
print(f'MLflow run_name: {MLFLOW_RUN_NAME}')

RUN_ID : 20260326_lightgbm_baseline_3d315dfc
RUN_PK : RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v2#20260326_lightgbm_baseline_3d315dfc
MLflow run_name: titanic-survival-prediction__tuned-hjsong-v2__hjsong__lightgbm


In [ ]:
# 시스템 메트릭 자동 수집 (CPU/메모리/디스크, MLflow >= 2.8)
mlflow.enable_system_metrics_logging()

mlflow_run = mlflow.start_run(run_name=MLFLOW_RUN_NAME)
print(f'MLflow Run ID: {mlflow_run.info.run_id}')

## 5. 데이터 로드 (DynamoDB DATA#split 아이템)

In [21]:
# DATA#split 아이템 조회: entity_type = 'DATA#train' / 'DATA#validation' / 'DATA#test'
# csv_b64 필드: base64 인코딩된 CSV 원본 → decode → BytesIO → DataFrame

def load_split_from_ddb(split):
    resp      = table.get_item(Key={'experiment_id': EXP_PK, 'entity_type': f'DATA#{split}'})
    csv_bytes = base64.b64decode(resp['Item']['csv_b64'])
    return pd.read_csv(io.BytesIO(csv_bytes))

train_df = load_split_from_ddb('train')
test_df  = load_split_from_ddb('test')

try:
    valid_df = load_split_from_ddb('validation')
    print(f'validation: {len(valid_df)}rows')
except Exception:
    valid_df = None

print(f'train: {len(train_df)}rows')
print(f'test : {len(test_df)}rows')

validation: 418rows
train: 891rows
test : 418rows


## 6. 전처리 및 학습/검증 분할

In [22]:
def preprocess_data(df):
    df = df.copy()
    df['Age']      = df['Age'].fillna(df['Age'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Fare']     = df['Fare'].fillna(df['Fare'].median())
    df['FamilySize']    = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']       = (df['FamilySize'] == 1).astype(int)
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']
    df['Sex']           = df['Sex'].map({'male': 1, 'female': 0})
    df = pd.concat([df, pd.get_dummies(df['Embarked'], prefix='Embarked')], axis=1)
    numeric_cols  = model_config['features']['numeric_col']
    feature_cols  = numeric_cols + ['Sex', 'Pclass', 'FamilySize', 'IsAlone', 'FarePerPerson']
    feature_cols += [c for c in df.columns if c.startswith('Embarked_')]
    target_col    = model_config['features']['target_col']
    X = df[feature_cols]
    y = df[target_col] if target_col in df.columns else None
    return X, y, feature_cols

X_full, y_full, feature_cols = preprocess_data(train_df)
SEED        = model_config['data']['seed']
VALID_RATIO = model_config['data']['train_valid_split']
X_train, X_valid, y_train, y_valid = train_test_split(
    X_full, y_full, test_size=VALID_RATIO, random_state=SEED, stratify=y_full)
print(f'Train:{len(X_train)}  Valid:{len(X_valid)}  Features:{len(feature_cols)}')

Train:712  Valid:179  Features:11


## 7. 모델 학습

In [23]:
params = {
    'objective':        model_config['hyperparameters']['objective'],
    'boosting':         model_config['hyperparameters']['boosting'],
    'max_depth':        model_config['hyperparameters']['max_depth'],
    'learning_rate':    model_config['hyperparameters']['learning_rate'],
    'num_leaves':       model_config['hyperparameters']['num_leaves'],
    'feature_fraction': model_config['hyperparameters']['feature_fraction'],
    'bagging_fraction': model_config['hyperparameters']['bagging_fraction'],
    'bagging_freq':     model_config['hyperparameters']['bagging_freq'],
    'verbosity': -1, 'seed': SEED, 'metric': ['auc', 'binary_logloss'],
}
evals_result = {}
model = lgb.train(
    params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=model_config['hyperparameters']['num_iterations'],
    valid_sets=[lgb.Dataset(X_train, label=y_train), lgb.Dataset(X_valid, label=y_valid)],
    valid_names=['train', 'valid'],
    callbacks=[lgb.log_evaluation(50), lgb.record_evaluation(evals_result)],
)
COMPLETED_AT = datetime.now().isoformat()
print(f'학습 완료. Best iteration: {model.best_iteration}')

[50]	train's auc: 0.931091	train's binary_logloss: 0.344982	valid's auc: 0.855731	valid's binary_logloss: 0.428074
[100]	train's auc: 0.951367	train's binary_logloss: 0.284161	valid's auc: 0.864888	valid's binary_logloss: 0.428906
[150]	train's auc: 0.964613	train's binary_logloss: 0.249951	valid's auc: 0.86357	valid's binary_logloss: 0.441443
[200]	train's auc: 0.974726	train's binary_logloss: 0.219429	valid's auc: 0.861067	valid's binary_logloss: 0.450731
학습 완료. Best iteration: 0


In [24]:
# MLflow 파라미터 로그
mlflow.log_params(params)
mlflow.log_param('train_samples',  len(X_train))
mlflow.log_param('valid_samples',  len(X_valid))
mlflow.log_param('feature_count',  len(feature_cols))
mlflow.log_param('best_iteration', model.best_iteration)
print('[MLflow] params logged')

[MLflow] params logged


## 8. 모델 평가

In [25]:
def evaluate_model(model, X, y, name=''):
    proba = model.predict(X)
    pred  = (proba >= 0.5).astype(int)
    m = {
        'accuracy':  round(accuracy_score(y, pred), 4),
        'precision': round(precision_score(y, pred), 4),
        'recall':    round(recall_score(y, pred), 4),
        'f1_score':  round(f1_score(y, pred), 4),
        'auc_roc':   round(roc_auc_score(y, proba), 4),
        'log_loss':  round(log_loss(y, proba), 4),
        'samples':   int(len(y)),
    }
    print(f'{name}: accuracy={m["accuracy"]}  auc={m["auc_roc"]}')
    return m, pred, proba

train_metrics, y_train_pred, y_train_proba = evaluate_model(model, X_train, y_train, 'Train')
valid_metrics, y_valid_pred, y_valid_proba = evaluate_model(model, X_valid, y_valid, 'Valid')

cm           = confusion_matrix(y_valid, y_valid_pred)
tn, fp, fn, tp = cm.ravel()

feature_importance = (
    pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importance('gain')})
    .sort_values('importance', ascending=False)
)
feature_importance['importance'] = feature_importance['importance'] / feature_importance['importance'].sum()
feature_importance['rank'] = range(1, len(feature_importance)+1)
print(feature_importance.head(5).to_string(index=False))

Train: accuracy=0.9171  auc=0.9747
Valid: accuracy=0.8324  auc=0.8611
      feature  importance  rank
          Sex    0.278120     1
          Age    0.198412     2
         Fare    0.195624     3
FarePerPerson    0.176970     4
       Pclass    0.068942     5


In [26]:
# MLflow 지표 로그
for k, v in valid_metrics.items():
    if isinstance(v, (int, float)):
        mlflow.log_metric(f'valid_{k}', v)
for k, v in train_metrics.items():
    if isinstance(v, (int, float)):
        mlflow.log_metric(f'train_{k}', v)
print('[MLflow] metrics logged')

[MLflow] metrics logged


## 9. 결과 저장 → DynamoDB

### 저장 아이템 목록
```
entity_type = CONFIG        → config 스냅샷
entity_type = CHARTS        → PNG 차트 4개 (base64 map)
entity_type = EXPLAINABILITY→ feature impact PNG (base64)
entity_type = MODEL#chunk_n → model.pkl (base64 청킹, 250KB/item)
entity_type = METRICS       → 학습 지표
entity_type = DATA_REF      → 데이터 출처 참조
entity_type = MANIFEST      → 실행 컨텍스트 전체
entity_type = REPORT        → HTML report
```

### 9.1 CONFIG 저장

In [27]:
table.put_item(Item=to_ddb({
    'experiment_id':  RUN_PK,
    'entity_type':    'CONFIG',
    'experiment_key': EXP_PK,
    'env_yml':        env_config,
    'meta_yml':       meta_config,
    'model_yml':      model_config,
    'saved_at':       CREATED_AT,
}))
print(f'[DDB] CONFIG 저장  →  experiment_id={RUN_PK}')

[DDB] CONFIG 저장  →  experiment_id=RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v2#20260326_lightgbm_baseline_3d315dfc


### 9.2 차트 생성 (메모리) + DDB/MLflow 저장

In [28]:
charts_bytes = {}

# Feature Importance
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feature_importance.head(10), x='importance', y='feature', palette='Blues_d', ax=ax)
ax.set_title('Feature Importance (Top 10)'); plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['feature_importance'] = buf.getvalue()

# ROC Curve
fpr, tpr, _ = roc_curve(y_valid, y_valid_proba)
auc_score   = roc_auc_score(y_valid, y_valid_proba)
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, 'b-', lw=2, label=f'AUC={auc_score:.4f}'); ax.plot([0,1],[0,1],'k--')
ax.fill_between(fpr, tpr, alpha=0.3); ax.set_title('ROC Curve'); ax.legend(); plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['roc_curve'] = buf.getvalue()

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Died','Survived'], yticklabels=['Died','Survived'])
ax.set_title('Confusion Matrix'); plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['confusion_matrix'] = buf.getvalue()

# Learning Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(evals_result['train']['auc'], label='Train'); axes[0].plot(evals_result['valid']['auc'], label='Valid')
axes[0].set_title('AUC'); axes[0].legend()
axes[1].plot(evals_result['train']['binary_logloss'], label='Train'); axes[1].plot(evals_result['valid']['binary_logloss'], label='Valid')
axes[1].set_title('Log Loss'); axes[1].legend(); plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['learning_curve'] = buf.getvalue()

# Feature Impact (Explainability)
top10 = feature_importance.head(10)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top10)), top10['importance'].values)
ax.set_yticks(range(len(top10))); ax.set_yticklabels(top10['feature'].values); ax.invert_yaxis()
ax.set_title('Feature Impact Summary'); plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['feature_impact_summary'] = buf.getvalue()

# DDB: training 4개 → CHARTS, explainability → EXPLAINABILITY
training_names = {'feature_importance', 'roc_curve', 'confusion_matrix', 'learning_curve'}
now = datetime.utcnow().isoformat()

table.put_item(Item={
    'experiment_id':  RUN_PK,
    'entity_type':    'CHARTS',
    'experiment_key': EXP_PK,
    'charts': {n: base64.b64encode(b).decode('utf-8')
               for n, b in charts_bytes.items() if n in training_names},
    'saved_at': now,
})
table.put_item(Item={
    'experiment_id':  RUN_PK,
    'entity_type':    'EXPLAINABILITY',
    'experiment_key': EXP_PK,
    'charts': {n: base64.b64encode(b).decode('utf-8')
               for n, b in charts_bytes.items() if n not in training_names},
    'saved_at': now,
})
print(f'[DDB] CHARTS / EXPLAINABILITY 저장 완료')

# MLflow: 임시 폴더 경유 업로드 후 삭제
tmp_dir = tempfile.mkdtemp()
try:
    for name, bts in charts_bytes.items():
        with open(os.path.join(tmp_dir, f'{name}.png'), 'wb') as f:
            f.write(bts)
    mlflow.log_artifacts(tmp_dir, artifact_path='charts')
    print('[MLflow] charts logged')
finally:
    shutil.rmtree(tmp_dir)

[DDB] CHARTS / EXPLAINABILITY 저장 완료
[MLflow] charts logged


### 9.3 모델 저장 (base64 청킹)

DynamoDB 아이템 최대 크기는 400KB입니다.
LightGBM model.pkl은 이를 초과할 수 있으므로 250KB 단위로 분할해서 저장합니다.

```
entity_type = MODEL#chunk_000  (0번째 청크)
entity_type = MODEL#chunk_001  (1번째 청크)
...
```
읽을 때는 chunk_index 순서로 조합 → base64 decode → pickle.loads

In [29]:
CHUNK_SIZE = 250_000   # base64 문자 단위 (~187KB raw)

pkl_bytes    = pickle.dumps(model)
b64_str      = base64.b64encode(pkl_bytes).decode('utf-8')
total_chunks = math.ceil(len(b64_str) / CHUNK_SIZE)
pkl_size_mb  = len(pkl_bytes) / (1024*1024)

print(f'model.pkl: {pkl_size_mb:.2f}MB → base64 {len(b64_str)/1024:.1f}KB → {total_chunks}개 청크')

now = datetime.utcnow().isoformat()
for i in range(total_chunks):
    chunk_data = b64_str[i * CHUNK_SIZE: (i+1) * CHUNK_SIZE]
    table.put_item(Item={
        'experiment_id':  RUN_PK,
        'entity_type':    f'MODEL#chunk_{i:03d}',   # 정렬을 위해 3자리 0패딩
        'experiment_key': EXP_PK,
        'chunk_index':    i,
        'total_chunks':   total_chunks,
        'algorithm':      ALGORITHM,
        'suffix':         SUFFIX,
        'format':         'pickle_base64',
        'data':           chunk_data,
        'saved_at':       now,
    })
    print(f'  [DDB] MODEL#chunk_{i:03d}  ({len(chunk_data)/1024:.1f}KB)')

# MLflow 모델 등록
mlflow.lightgbm.log_model(model, artifact_path='model')
print('[MLflow] model logged')

2026/03/26 11:48:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


model.pkl: 0.33MB → base64 453.6KB → 2개 청크
  [DDB] MODEL#chunk_000  (244.1KB)
  [DDB] MODEL#chunk_001  (209.4KB)
[MLflow] model logged


### 9.4 METRICS 저장

In [30]:
model_metrics = {
    'run_id':       RUN_ID,
    'model_name':   meta_config['model'],
    'algorithm':    ALGORITHM,
    'task':         'binary_classification',
    'evaluated_at': COMPLETED_AT,
    'train_metrics': train_metrics,
    'valid_metrics': valid_metrics,
    'confusion_matrix': {
        'true_negative':  int(tn), 'false_positive': int(fp),
        'false_negative': int(fn), 'true_positive':  int(tp),
        'specificity': round(float(tn/(tn+fp)), 4),
        'sensitivity': round(float(tp/(tp+fn)), 4),
    },
    'feature_importance': [
        {'feature': r['feature'], 'importance': round(float(r['importance']),4), 'rank': int(r['rank'])}
        for _, r in feature_importance.head(10).iterrows()
    ],
    'training_history': {
        'best_iteration':  model.best_iteration,
        'final_train_auc': round(evals_result['train']['auc'][-1], 4),
        'final_valid_auc': round(evals_result['valid']['auc'][-1], 4),
    },
    'model_complexity': {
        'num_trees': model.num_trees(), 'max_depth': params['max_depth'],
        'num_leaves': params['num_leaves'], 'total_features': len(feature_cols),
        'pkl_size_mb': round(pkl_size_mb, 2),
    },
}
table.put_item(Item=to_ddb({
    'experiment_id':  RUN_PK,
    'entity_type':    'METRICS',
    'experiment_key': EXP_PK,
    **model_metrics,
}))
print(f'[DDB] METRICS 저장 완료')

[DDB] METRICS 저장 완료


### 9.5 DATA_REF 저장

In [31]:
table.put_item(Item=to_ddb({
    'experiment_id':  RUN_PK,
    'entity_type':    'DATA_REF',
    'experiment_key': EXP_PK,
    'source': {
        'type':    'dynamodb',
        'table':   TABLE_NAME,
        'exp_pk':  EXP_PK,
        'version': VERSION,
    },
    'split_keys': {
        'train':      f'{EXP_PK}|DATA#train',
        'validation': f'{EXP_PK}|DATA#validation',
        'test':       f'{EXP_PK}|DATA#test',
    },
    'schema': {
        'target_col':    model_config['features']['target_col'],
        'feature_count': len(feature_cols),
        'feature_cols':  feature_cols,
    },
    'split': {
        'method':            'stratified_random',
        'train_ratio':       1 - VALID_RATIO,
        'valid_ratio':       VALID_RATIO,
        'seed':              SEED,
        'train_samples':     int(len(X_train)),
        'valid_samples':     int(len(X_valid)),
        'train_target_dist': {str(k): int(v) for k,v in y_train.value_counts().sort_index().items()},
        'valid_target_dist': {str(k): int(v) for k,v in y_valid.value_counts().sort_index().items()},
    },
}))
print(f'[DDB] DATA_REF 저장 완료')

[DDB] DATA_REF 저장 완료


### 9.6 MANIFEST 저장

In [32]:
import sys as _sys
duration = int((datetime.fromisoformat(COMPLETED_AT) - datetime.fromisoformat(CREATED_AT)).total_seconds())

table.put_item(Item=to_ddb({
    'experiment_id':  RUN_PK,
    'entity_type':    'MANIFEST',
    'experiment_key': EXP_PK,
    'run_id':       RUN_ID,
    'run_name':     f'{EXPERIMENT}-{ALGORITHM}-{SUFFIX}',
    'created_at':   CREATED_AT,
    'completed_at': COMPLETED_AT,
    'status':       'completed',
    'context': {
        'env': ENV, 'user_id': USER_ID, 'project': PROJECT,
        'experiment': EXPERIMENT, 'model': meta_config['model'],
        'algorithm': ALGORITHM, 'version': VERSION,
    },
    'storage': {'type': 'dynamodb', 'table': TABLE_NAME},
    'runtime': {
        'platform': 'local',
        'python_version': f'{_sys.version_info.major}.{_sys.version_info.minor}',
        'framework': f'lightgbm=={lgb.__version__}',
    },
    'summary': {
        'duration_seconds': duration,
        'train_samples':    int(len(X_train)),
        'valid_samples':    int(len(X_valid)),
        'feature_count':    len(feature_cols),
        'best_iteration':   model.best_iteration,
    },
}))
print(f'[DDB] MANIFEST 저장 완료')

[DDB] MANIFEST 저장 완료


### 9.7 REPORT 저장 (HTML)

In [33]:
html_report = f"""<!DOCTYPE html><html lang="ko"><head>
<meta charset="UTF-8"><title>Report - {RUN_ID}</title>
<style>body{{font-family:Arial;margin:40px}} .grid{{display:grid;grid-template-columns:repeat(4,1fr);gap:20px}}
.card{{background:#f8f9fa;padding:20px;border-radius:8px;text-align:center}}
.val{{font-size:2em;font-weight:bold;color:#2E75B6}}
table{{width:100%;border-collapse:collapse}} th{{background:#2E75B6;color:white;padding:10px}}
td{{padding:10px;border-bottom:1px solid #ddd}}</style></head>
<body><h1>Titanic Classification Report</h1>
<p><b>Run:</b> {RUN_ID} | <b>Experiment:</b> {EXPERIMENT} | <b>Algorithm:</b> {ALGORITHM}</p>
<h2>Validation Metrics</h2>
<div class="grid">
  <div class="card"><div class="val">{valid_metrics['accuracy']:.1%}</div>Accuracy</div>
  <div class="card"><div class="val">{valid_metrics['precision']:.1%}</div>Precision</div>
  <div class="card"><div class="val">{valid_metrics['recall']:.1%}</div>Recall</div>
  <div class="card"><div class="val">{valid_metrics['auc_roc']:.4f}</div>AUC-ROC</div>
</div>
<h2>Feature Importance (Top 10)</h2>
<table><tr><th>Rank</th><th>Feature</th><th>Importance</th></tr>
{''.join(f"<tr><td>{int(r['rank'])}</td><td>{r['feature']}</td><td>{r['importance']:.4f}</td></tr>" for _,r in feature_importance.head(10).iterrows())}
</table></body></html>"""

table.put_item(Item={
    'experiment_id':  RUN_PK,
    'entity_type':    'REPORT',
    'experiment_key': EXP_PK,
    'format':         'html',
    'content':        html_report,
    'saved_at':       datetime.utcnow().isoformat(),
})
print(f'[DDB] REPORT 저장 완료')

[DDB] REPORT 저장 완료


## 10. MLflow 태그 + 종료

In [34]:
mlflow.set_tag('run_id',     RUN_ID)
mlflow.set_tag('run_name',   MLFLOW_RUN_NAME)
mlflow.set_tag('user_id',    USER_ID)
mlflow.set_tag('project',    PROJECT)
mlflow.set_tag('experiment', EXPERIMENT)
mlflow.set_tag('algorithm',  ALGORITHM)
mlflow.set_tag('env',        ENV)
for tag in MLFLOW_TAGS:
    mlflow.set_tag(tag, 'true')

mlflow.end_run()
print('[MLflow] 등록 완료')
print(f'  Run Name  : {MLFLOW_RUN_NAME}')
print(f'  Run ID    : {mlflow_run.info.run_id}')
print(f'  Experiment: {MLFLOW_EXPERIMENT}')

🏃 View run titanic-survival-prediction__tuned-hjsong-v2__hjsong__lightgbm at: http://edu-mlflow-alb-697692344.us-east-1.elb.amazonaws.com/#/experiments/5/runs/70a43c4143b64b6cab4534dcd17648cd
🧪 View experiment at: http://edu-mlflow-alb-697692344.us-east-1.elb.amazonaws.com/#/experiments/5


2026/03/26 11:49:15 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/26 11:49:17 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


[MLflow] 등록 완료
  Run Name  : titanic-survival-prediction__tuned-hjsong-v2__hjsong__lightgbm
  Run ID    : 70a43c4143b64b6cab4534dcd17648cd
  Experiment: passenger-survival-classifier/tuned-hjsong-v2


## 11. 검증 (DynamoDB 저장 확인)

In [35]:
print(f'RUN PK: {RUN_PK}')
print('=' * 70)
resp  = table.query(KeyConditionExpression=Key('experiment_id').eq(RUN_PK))
for item in sorted(resp['Items'], key=lambda x: x['entity_type']):
    size_kb = len(str(item)) / 1024
    print(f'  entity_type={item["entity_type"]:<30}  ~{size_kb:.1f}KB')
print('=' * 70)
print(f'총 {len(resp["Items"])}개 아이템')

RUN PK: RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v2#20260326_lightgbm_baseline_3d315dfc
  entity_type=CHARTS                          ~293.0KB
  entity_type=CONFIG                          ~3.6KB
  entity_type=DATA_REF                        ~1.1KB
  entity_type=EXPLAINABILITY                  ~54.4KB
  entity_type=MANIFEST                        ~1.0KB
  entity_type=METRICS                         ~2.2KB
  entity_type=MODEL#chunk_000                 ~244.5KB
  entity_type=MODEL#chunk_001                 ~209.8KB
  entity_type=REPORT                          ~1.9KB
총 9개 아이템


## 12. 최종 요약

In [36]:
print('=' * 60)
print(f'Run ID : {RUN_ID}')
print(f'EXP_PK : {EXP_PK}')
print(f'RUN_PK : {RUN_PK}')
print()
print(f'Valid Accuracy : {valid_metrics["accuracy"]:.4f}')
print(f'Valid AUC-ROC  : {valid_metrics["auc_roc"]:.4f}')
print()
print('모델 로드 방법:')
print(f'  # DDB에서 청크 조합 후 복원')
print(f'  resp  = table.query(KeyConditionExpression=Key("experiment_id").eq(RUN_PK) & Key("entity_type").begins_with("MODEL#chunk_"))')
print(f'  items = sorted(resp["Items"], key=lambda x: int(x["chunk_index"]))')
print(f'  model = pickle.loads(base64.b64decode("".join(i["data"] for i in items)))')
print(f'완료: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

Run ID : 20260326_lightgbm_baseline_3d315dfc
EXP_PK : EXP#hjsong#titanic-survival-prediction#tuned-hjsong-v2
RUN_PK : RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v2#20260326_lightgbm_baseline_3d315dfc

Valid Accuracy : 0.8324
Valid AUC-ROC  : 0.8611

모델 로드 방법:
  # DDB에서 청크 조합 후 복원
  resp  = table.query(KeyConditionExpression=Key("experiment_id").eq(RUN_PK) & Key("entity_type").begins_with("MODEL#chunk_"))
  items = sorted(resp["Items"], key=lambda x: int(x["chunk_index"]))
  model = pickle.loads(base64.b64decode("".join(i["data"] for i in items)))
완료: 2026-03-26 11:49:25


In [37]:
# resp  = table.query(KeyConditionExpression=Key("experiment_id").eq(RUN_PK) & Key("entity_type").begins_with("MODEL#chunk_"))
# items = sorted(resp["Items"], key=lambda x: int(x["chunk_index"]))
# model = pickle.loads(base64.b64decode("".join(i["data"] for i in items)))